In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: دالة Qiskit من Qedma
*راجع [مرجع واجهة برمجة التطبيقات](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** دوال Qiskit ميزة تجريبية متاحة فقط لمستخدمي خطط IBM Quantum&reg; Premium وFlex وOn-Prem (عبر IBM Quantum Platform API). وهي في مرحلة إصدار معاينة وقابلة للتغيير.

## نظرة عامة
على الرغم من التحسينات الكبيرة التي شهدتها وحدات المعالجة الكمومية في السنوات الأخيرة، لا تزال الأخطاء الناجمة عن الضوضاء والعيوب في الأجهزة الحالية تمثّل تحديًا محوريًا لمطوّري الخوارزميات الكمومية. ومع اقتراب المجال من الحسابات الكمومية ذات المقياس الأداتي التي يتعذّر التحقق منها كلاسيكيًا، تتزايد أهمية الحلول القادرة على إلغاء الضوضاء بدقة مضمونة. للتغلب على هذا التحدي، طوّرت Qedma نظام تخفيف الأخطاء الكمومي (QESEM)، المدمج بسلاسة في IBM Quantum Platform بوصفه [دالة Qiskit](/guides/functions).

مع QESEM، يمكن للمستخدمين تشغيل دوائرهم الكمومية على وحدات المعالجة الكمومية الصاخبة للحصول على نتائج دقيقة للغاية وخالية من الأخطاء، مع تكاليف زمنية منخفضة وقريبة من الحدود الأساسية. لتحقيق ذلك، يستخدم QESEM مجموعة من الأساليب الخاصة التي طوّرتها Qedma لتوصيف الأخطاء وتقليلها. تشمل تقنيات تقليل الأخطاء: تحسين البوابات، والترانسبيلاسيون الواعي بالضوضاء، وقمع الأخطاء (ES)، وتخفيف الأخطاء غير المتحيز (EM). بهذا الجمع من الأساليب القائمة على التوصيف، يستطيع المستخدمون الحصول على نتائج موثوقة وخالية من الأخطاء للدوائر الكمومية الكبيرة ذات الأحجام الكبيرة، مما يفتح تطبيقات لا يمكن تحقيقها بطريقة أخرى.

للاطلاع على وصف كامل للمكوّنات الأساسية، إضافةً إلى عرض توضيحي بمقياس أداتي، راجع الورقة البحثية [تخفيف الأخطاء بدقة عالية وموثوقة للدوائر الكمومية بمقياس الأداة](https://arxiv.org/abs/2508.10997).

## الوصف
يمكنك استخدام دالة QESEM من Qedma لتقدير تكلفة تنفيذ دوائرك وتشغيلها بسهولة مع قمع الأخطاء وتخفيفها، مما يتيح أحجام دوائر أكبر ودقة أعلى. لاستخدام QESEM، تُقدّم دارة كمومية، ومجموعة من الالمؤثرات (observables) لقياسها، ودقة إحصائية مستهدفة لكل رصيدة، ووحدة معالجة كمومية مختارة. قبل تشغيل الدارة للوصول إلى الدقة المستهدفة، يمكنك تقدير وقت وحدة المعالجة الكمومية المطلوب استنادًا إلى حساب تحليلي لا يستلزم تنفيذ الدارة. بمجرد أن تقتنع بتقدير وقت وحدة المعالجة الكمومية، يمكنك تشغيل الدارة مع QESEM.

عند تشغيل دارة ما، يُنفّذ QESEM بروتوكول توصيف الجهاز المُخصَّص لدارتك، مما يُنتج نموذج ضوضاء موثوقًا للأخطاء التي تحدث في الدارة. استنادًا إلى هذا التوصيف، يُطبّق QESEM أولًا الترانسبيلاسيون الواعي بالضوضاء لتعيين الدارة المُدخَلة على مجموعة من الكيوبتات والبوابات الفيزيائية، مما يُقلّل من الضوضاء المؤثرة على المؤثر المستهدفة. وتشمل هذه البوابات المتاحة أصلًا (CX/CZ على أجهزة IBM&reg;)، إضافةً إلى بوابات إضافية مُحسَّنة بواسطة QESEM، لتشكيل مجموعة البوابات الموسّعة الخاصة بـ QESEM. ثم يُشغّل QESEM مجموعة من دوائر ES وEM القائمة على التوصيف على وحدة المعالجة الكمومية ويجمع نتائج قياساتها. تُعالَج هذه النتائج بعد ذلك كلاسيكيًا لتوفير قيمة توقع غير متحيزة وهامش خطأ لكل رصيدة، مقابلةً للدقة المطلوبة.

![نظرة عامة على Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
أثبت QESEM قدرته على تقديم نتائج عالية الدقة لمجموعة متنوعة من التطبيقات الكمومية وعلى أكبر أحجام الدوائر القابلة للتحقيق اليوم. يتيح QESEM المميزات التالية الموجّهة للمستخدم، والمستعرضة في قسم المعايير أدناه:
-	**دقة مضمونة:** يُخرج QESEM تقديرات غير متحيزة لقيم التوقع للالمؤثرات. أسلوب EM الخاص به مجهّز بضمانات نظرية، التي - جنبًا إلى جنب مع توصيف Qedma المتقدم - تضمن تقارب التخفيف نحو مخرجات الدارة الخالية من الضوضاء وصولًا إلى الدقة المحددة من قِبَل المستخدم. على النقيض من كثير من أساليب EM الاستدلالية المعرضة للأخطاء المنهجية أو التحيّزات، فإن دقة QESEM المضمونة ضرورية لضمان نتائج موثوقة في الدوائر والالمؤثرات الكمومية العامة.
-	**قابلية التوسّع لوحدات معالجة كمومية كبيرة:** يعتمد وقت وحدة المعالجة الكمومية في QESEM على أحجام الدوائر، لكنه مستقل في حالات أخرى عن عدد الكيوبتات. أثبتت Qedma عمل QESEM على أكبر الأجهزة الكمومية المتاحة اليوم، بما فيها أجهزة IBM Quantum Eagle بـ 127 كيوبت وHeron بـ 133 كيوبت.
-	**لا يختص بتطبيق بعينه:** جرى توظيف QESEM في مجموعة متنوعة من التطبيقات، تشمل محاكاة هاميلتونيان، وVQE، وQAOA، وتقدير السعة. يمكن للمستخدمين إدخال أي دارة كمومية ومؤثر لقياسها والحصول على نتائج دقيقة وخالية من الأخطاء. القيود الوحيدة تفرضها مواصفات الجهاز ووقت وحدة المعالجة الكمومية المخصص، اللذان يحددان أحجام الدوائر المتاحة ودقة المخرجات. في المقابل، كثير من حلول تقليل الأخطاء مخصصة لتطبيقات بعينها أو تعتمد على إرشاديات غير مضبوطة، مما يجعلها غير صالحة للدوائر والتطبيقات الكمومية العامة.
-  **مجموعة بوابات موسّعة:** يدعم QESEM بوابات الزوايا الكسرية، ويوفّر بوابات $Rzz(\theta)$ ذات الزوايا الكسرية المُحسَّنة من Qedma على أجهزة IBM Quantum Heron وEagle. تُتيح مجموعة البوابات الموسّعة هذه تصريفًا أكثر كفاءة وتفتح أحجام دوائر أكبر بعامل يصل إلى 2 مقارنةً بتصريف CX/CZ الافتراضي.
-	**المؤثرات متعددة القواعد:** يدعم QESEM الالمؤثرات المُدخَلة المؤلفة من عديد من سلاسل باولي غير المتبادلة في التبديل، كهاميلتونيانات عامة. يتم اختيار قواعد القياس وتحسين توزيع موارد وحدة المعالجة الكمومية (اللقطات والدوائر) تلقائيًا بواسطة QESEM لتقليل وقت وحدة المعالجة الكمومية المطلوب لتحقيق الدقة المطلوبة. يأخذ هذا التحسين في الاعتبار نسب الإخلاص ومعدلات التنفيذ للجهاز، مما يُمكّنك من تشغيل دوائر أعمق والحصول على دقة أعلى.

## المعايير
جرى اختبار QESEM على مجموعة واسعة من حالات الاستخدام والتطبيقات. يمكن للأمثلة التالية مساعدتك في تقييم أنواع أعباء العمل التي يمكنك تشغيلها مع QESEM.

مؤشر رئيسي لقياس صعوبة كل من تخفيف الأخطاء والمحاكاة الكلاسيكية لدارة ومؤثر معينة هو **الحجم الفعّال**: عدد بوابات CNOT المؤثرة على المؤثر في الدارة. يعتمد الحجم الفعّال على عمق الدارة وعرضها، ووزن المؤثر، وبنية الدارة التي تُحدد المخروط الضوئي للمؤثر. لمزيد من التفاصيل، راجع الحديث من [قمة IBM Quantum 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). يُقدّم QESEM قيمة كبيرة بصفة خاصة في نظام الحجم الكبير، إذ يُعطي نتائج موثوقة للدوائر والالمؤثرات العامة.

![الحجم الفعّال](../docs/images/guides/qedma-qesem/active_volume.svg)

| التطبيق | عدد الكيوبتات | الجهاز | وصف الدارة | الدقة | الوقت الإجمالي | استخدام وقت التشغيل |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| دارة VQE  | 8              | Eagle (r3) | 21 طبقة إجمالية، 9 قواعد قياس، سلسلة أحادية الأبعاد                    | 98%      | 35 دقيقة      | 14 دقيقة         |
| Kicked Ising   | 28              | Eagle (r3) | 3 طبقات فريدة × 3 خطوات، هيكلية hex ثنائية الأبعاد                      | 97%     | 22 دقيقة      | 4 دقائق          |
| Kicked Ising   | 28              | Eagle (r3) | 3 طبقات فريدة × 8 خطوات، هيكلية hex ثنائية الأبعاد                     | 97%      | 116 دقيقة      | 23 دقيقة          |
| محاكاة هاميلتونيان بطريقة تروتر | 40  | Eagle (r3)            | 2 طبقتان فريدتان × 10 خطوات تروتر، سلسلة أحادية الأبعاد                    | 97%      | 3 ساعات     | 25 دقيقة         |
| محاكاة هاميلتونيان بطريقة تروتر | 119  | Eagle (r3)           | 3 طبقات فريدة × 9 خطوات تروتر، هيكلية hex ثنائية الأبعاد                    | 95%      | 6.5 ساعات     | 45 دقيقة         |
| Kicked Ising  | 136             | Heron (r2) | 3 طبقات فريدة × 15 خطوة، هيكلية hex ثنائية الأبعاد                    | 99%      | 52 دقيقة             | 9 دقائق           |

تُقاس الدقة هنا بالنسبة إلى القيمة المثالية للمؤثر: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$، حيث '$\epsilon$' هي الدقة المطلقة للتخفيف (التي يحددها المستخدم)، و$\langle O \rangle_{ideal}$ هي المؤثر في الدارة الخالية من الضوضاء.
يقيس "استخدام وقت التشغيل" استخدام المعيار في وضع الدُّفعات (مجموع استخدام الوظائف الفردية)، بينما يقيس "الوقت الإجمالي" الاستخدام في وضع الجلسة (وقت الجدار التجريبي)، الذي يشمل الأوقات الكلاسيكية وأوقات الاتصال الإضافية. يتوفر QESEM للتنفيذ في كلا الوضعين، بحيث يمكن للمستخدمين الاستفادة القصوى من مواردهم المتاحة.

تُحاكي دوائر Kicked Ising ذات 28 كيوبتًا بلّورة الوقت المتقطعة الشبه دورية (Discrete Time Quasicrystal) التي درسها Shinjo وآخرون (انظر [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) و[Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) على ثلاث حلقات متصلة من ibm_kawasaki. معاملات الدارة المستخدمة هنا هي $(\theta_x, \theta_z) = (0.9 \pi, 0)$، مع حالة ابتدائية مغناطيسية $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. المؤثر المقيسة هي القيمة المطلقة للمغنطة $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. جرى تشغيل تجربة Kicked Ising ذات المقياس الأداتي على أفضل 136 كيوبت من ibm_fez؛ أُجري هذا المعيار تحديدًا عند زاوية كليفورد $(\theta_x, \theta_z) = (\pi, 0)$، حيث يتنامى الحجم الفعّال ببطء مع عمق الدارة، مما يُتيح - مقرونًا بنسب إخلاص الجهاز العالية - تحقيق دقة عالية في وقت تشغيل قصير.

دوائر محاكاة هاميلتونيان بطريقة تروتر مخصصة لنموذج Ising بالحقل المستعرض عند زوايا كسرية: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ و$(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ على التوالي (انظر [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). جرى تشغيل الدارة ذات المقياس الأداتي على أفضل 119 كيوبت من ibm_brisbane، بينما أُجريت تجربة الـ 40 كيوبتًا على أفضل سلسلة متاحة. تُبلَّغ الدقة للمغنطة؛ وقد تم الحصول على نتائج عالية الدقة للالمؤثرات ذات الأوزان الأعلى أيضًا.

طُوّرت دارة VQE بالتعاون مع باحثين من مركز تكنولوجيا الكم والتطبيقات في مركز سينكروترون الإلكترونات الألماني (DESY). المؤثر المستهدفة هنا كانت هاميلتونيانًا يتألف من عدد كبير من سلاسل باولي غير المتبادلة في التبديل، مما يبرز الأداء المُحسَّن لـ QESEM في الالمؤثرات متعددة القواعد. طُبّق التخفيف على نموذج مُحسَّن كلاسيكيًا؛ وعلى الرغم من أن هذه النتائج لا تزال غير منشورة، فإن نتائج بنفس الجودة ستُحصَّل لدوائر مختلفة ذات خصائص بنيوية مماثلة.

## البدء
سجّل الدخول باستخدام [مفتاح IBM Quantum Platform API](http://quantum.cloud.ibm.com/)، واختر دالة QESEM Qiskit كما يلي. (يفترض هذا المقتطف أنك قد [حفظت حسابك](/guides/functions#install-qiskit-functions-catalog-client) بالفعل في بيئتك المحلية.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog


catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [3]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [4]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

يمكنك استخدام واجهات برمجة تطبيقات Qiskit Serverless المألوفة للتحقق من حالة عبء عمل دالة Qiskit أو استرداد النتائج:

In [5]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

يصف مقتطف الكود التالي كيفية استرداد تقدير وقت وحدة المعالجة الكمومية (عند ضبط `estimate_time_only`):

In [6]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)

print(sample_job.status())
result = sample_job.result()

9a87a23f-82f5-429e-91fb-cc8a9d107980
QUEUED


يُظهر مقتطف الكود التالي كيفية استرداد نتائج التخفيف (عند عدم ضبط `estimate_time_only`) ومقاييس التنفيذ. تحتوي هذه على بيانات أساسية تُتيح فهمًا أعمق لكيفية تأثير المعاملات المختلفة على تنفيذ QESEM. قد تكون ذات صلة أيضًا عند كتابة ورقة بحثية مستندة إلى أبحاثك.

In [7]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

The estimated QPU time for this PUB is: 
{'time_estimation_sec': 1800, 'description': None, 'instance': 'crn:v1:bluemix:public:quantum-computing:us-east:a/6c63dae5281147f1a0449b36e0aaba3a:ae40ab55-8c55-4042-9204-71a6541d56ec::', 'transpilation_level': 'standard', 'parallel_execution': False, 'total_qpu_time': 0, 'empirical_estimation_mitigation_results': None, 'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 42.44837867887691, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 17.879877626895905, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}}


## استرداد رسائل الخطأ
إذا كانت حالة عبء عملك هي ERROR، استخدم `job.result()` لاسترداد رسالة الخطأ كما يلي:

In [9]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    if "qubit_map" in circuit:
        print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

Mitigated expectation values: 
[1.00169764 1.00460812]
Mitigated error-bar: 
[0.00155021 0.0099558 ]
Noisy expectation values: 
[0.95717143 0.94271429]
Noisy error-bar: 
[0.00206982 0.00872689]
Total QPU time: 
 150.0
Gates fidelity measured during the experiment: 
 {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}
Total shots / mitigation shots: 
 495600 / 220400
Transpiled circuits:
Circuit 0:
  Circuit: 
 OPENQASM 3.0;
include "stdgates.inc";
bit[145] c0;
qubit[145] q0;
rz(-pi) q0[143];
rz(0) q0[141];
rz(-pi) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(-pi/2) q0[143];
rz(pi/2) q0[141];
rz(-pi/2) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(pi/2) q0[143];
rz(-pi/2) q0[141];
rz(pi/2) q0[140];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
cz q0[144], q0[143];
cz q0[142], q0[141];
barrier q0[144], q0[143], q0[142], q0[141];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
rz(-pi) q0[144];
rz(-pi/2) q0[143];
rz(0) q0[142];
rz(-pi/2) q0[141];
sx q0[144];
sx q0[143];

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## الحصول على الدعم

فريق دعم Qedma هنا لمساعدتك! إذا واجهت أي مشكلات أو لديك أسئلة حول استخدام دالة QESEM Qiskit، لا تتردد في التواصل معنا. موظفو الدعم المتمرسون والودودون لدينا مستعدون لمساعدتك في أي مخاوف أو استفسارات تقنية قد تكون لديك.

يمكنك مراسلتنا عبر البريد الإلكتروني على support@qedma.com للحصول على المساعدة. يُرجى تضمين أكبر قدر ممكن من التفاصيل حول المشكلة التي تواجهها لمساعدتنا في تقديم استجابة سريعة ودقيقة. يمكنك أيضًا التواصل مع ممثل Qedma POC المخصص لك عبر البريد الإلكتروني أو الهاتف.

لمساعدتنا في خدمتك بكفاءة أكبر، يُرجى تقديم المعلومات التالية عند التواصل معنا:

- وصف تفصيلي للمشكلة
- معرّف الوظيفة (Job ID)
- أي رسائل خطأ أو رموز ذات صلة

نحن ملتزمون بتزويدك بدعم سريع وفعّال لضمان حصولك على أفضل تجربة ممكنة مع دالة Qiskit الخاصة بنا.

نحن نسعى دائمًا إلى تحسين منتجنا ونرحّب باقتراحاتك! إذا كان لديك أفكار حول كيفية تحسين خدماتنا أو ميزات تودّ رؤيتها، يُرجى إرسال أفكارك إلى support@qedma.com.

## الخطوات التالية

> **Tip:** - [اطلب الوصول إلى Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
> - تفضّل بزيارة [مرجع واجهة برمجة التطبيقات](https://docs.quantum.ibm.com/api/functions/qedma-qesem) لدالة Qiskit هذه.
> - راجع [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
> - راجع [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).